<a href="https://colab.research.google.com/github/petrovortex/foundations_of_ml_course/blob/main/applied_ML_%5BLSTM%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Провести анализ качества аппроксимации выборки NERUS (предсказание POS tag для токенов) моделью LSTM в зависимости от:

- размера слоя;
- числа слоев;
- параметра dropout;
- добавления BatchNorm;
- размера словаря;
- *токенизатора* - дополнительное задание (со звездочкой).

## Data

In [1]:
!pip install -q conllu nerus

In [ ]:
!wget https://storage.yandexcloud.net/natasha-nerus/data/nerus_lenta.conllu.gz

In [3]:
import gzip
from conllu import parse_incr
from nerus import load_nerus
from typing import List, Tuple

In [4]:
def load_nerus_data(path: str, limit: int = 50000) -> List[Tuple[List[str], List[str]]]:
    data = []
    docs = load_nerus(path)

    for doc in docs:
        for sent in doc.sents:
            if len(data) >= limit:
                return data

            words = [token.text for token in sent.tokens]
            tags = [token.pos for token in sent.tokens]

            data.append((words, tags))

    return data

nerus_data = load_nerus_data('nerus_lenta.conllu.gz', limit=50000)

## Dataset

In [5]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
from collections import Counter
from typing import List, Tuple, Dict, Any, Optional


class NERUSDataset(Dataset):
    def __init__(
        self,
        data: List[Tuple[List[str], List[str]]],
        tokenizer_type: str = "word",
        vocab_size: int = 10000
    ) -> None:
        self.data = data
        self.tokenizer_type = tokenizer_type
        self.ignore_index = -100
        self.pad_token_id = 0

        self.tag2idx = self._build_tag_vocab()

        if self.tokenizer_type == "word":
            self.word2idx = self._build_word_vocab(vocab_size)
        elif self.tokenizer_type == "labse":
            self.tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/LaBSE")
            self.word2idx = self.tokenizer.vocab
        elif self.tokenizer_type == "char":
            self.word2idx = self._build_char_vocab()

    def _build_tag_vocab(self) -> Dict[str, int]:
        unique_tags = set(tag for _, tags in self.data for tag in tags)
        return {tag: idx for idx, tag in enumerate(sorted(unique_tags))}

    def _build_word_vocab(self, vocab_size: int) -> Dict[str, int]:
        words = [word for words, _ in self.data for word in words]
        most_common = Counter(words).most_common(vocab_size)
        vocab = {word: idx + 1 for idx, (word, _) in enumerate(most_common)}
        vocab["<UNK>"] = 0
        return vocab

    def _build_char_vocab(self) -> Dict[str, int]:
        chars = set(char for words, _ in self.data for word in words for char in word)
        return {char: idx + 1 for idx, char in enumerate(sorted(chars))}

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        words, tags = self.data[idx]

        if self.tokenizer_type == "word":
            return self._get_word_tokens(words, tags)
        elif self.tokenizer_type == "labse":
            return self._get_labse_tokens(words, tags)
        return self._get_char_tokens(words, tags)

    def _get_word_tokens(self, words: List[str], tags: List[str]) -> Tuple[torch.Tensor, torch.Tensor]:
        input_ids = [self.word2idx.get(w, 0) for w in words]
        labels = [self.tag2idx[t] for t in tags]
        return torch.tensor(input_ids), torch.tensor(labels)

    def _get_char_tokens(self, words: List[str], tags: List[str]) -> Tuple[torch.Tensor, torch.Tensor]:
        input_ids = []
        labels = []

        for word, tag in zip(words, tags):
            word_chars = [self.word2idx.get(c, 0) for c in word]
            input_ids.extend(word_chars)

            word_labels = [self.tag2idx[tag]] + [self.ignore_index] * (len(word_chars) - 1)
            labels.extend(word_labels)

        return torch.tensor(input_ids), torch.tensor(labels)

    def _get_labse_tokens(self, words: List[str], tags: List[str]) -> Tuple[torch.Tensor, torch.Tensor]:
        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            return_offsets_mapping=True,
            add_special_tokens=True
        )

        input_ids = encoding["input_ids"]
        offset_mapping = encoding["offset_mapping"]
        labels = []

        word_idx = -1
        for i, (start, end) in enumerate(offset_mapping):
            if start == 0 and end == 0:
                labels.append(self.ignore_index)
            elif start == 0:
                word_idx += 1
                labels.append(self.tag2idx[tags[word_idx]])
            else:
                labels.append(self.ignore_index)

        return torch.tensor(input_ids), torch.tensor(labels)

    def calculate_oov_rate(self) -> float:
        if self.tokenizer_type != "word":
            return 0.0

        total_words = 0
        oov_words = 0

        for words, _ in self.data:
            for word in words:
                total_words += 1
                if word not in self.word2idx:
                    oov_words += 1

        return oov_words / total_words if total_words > 0 else 0.0

In [6]:
from torch.nn.utils.rnn import pad_sequence
from typing import List, Tuple

def collate_fn(batch: List[Tuple[torch.Tensor, torch.Tensor]]) -> Tuple[torch.Tensor, torch.Tensor]:
    inputs, labels = zip(*batch)

    padded_inputs = pad_sequence(inputs, batch_first=True, padding_value=0)
    padded_labels = pad_sequence(labels, batch_first=True, padding_value=-100)

    return padded_inputs, padded_labels

## LSTM & Trainer

In [7]:
import torch.nn as nn

In [8]:
class ConfigurableLSTM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int = 128,
        hidden_size: int = 256,
        num_layers: int = 1,
        dropout: float = 0.0,
        norm_type: str = "none"
    ) -> None:
        super().__init__()

        self.norm_type = norm_type

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False
        )

        if norm_type == "batch":
            self.norm = nn.BatchNorm1d(hidden_size)
        elif norm_type == "layer":
            self.norm = nn.LayerNorm(hidden_size)
        else:
            self.norm = nn.Identity()

        self.classifier = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.embedding(x)

        lstm_out, _ = self.lstm(x)

        if self.norm_type == "batch":
            lstm_out = lstm_out.permute(0, 2, 1)
            lstm_out = self.norm(lstm_out)
            lstm_out = lstm_out.permute(0, 2, 1)
        elif self.norm_type == "layer":
            lstm_out = self.norm(lstm_out)

        lstm_out = self.dropout(lstm_out)
        logits = self.classifier(lstm_out)

        return logits

In [9]:
from torch.utils.data import DataLoader
from torch.optim import Optimizer
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm

In [10]:
class Trainer:
    def __init__(
        self,
        model: nn.Module,
        optimizer: Optimizer,
        criterion: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        device: torch.device,
        writer: SummaryWriter,
        ignore_index: int = -100
    ) -> None:
        self.model = model.to(device)
        self.optimizer = optimizer
        self.criterion = criterion
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.writer = writer
        self.ignore_index = ignore_index

    def _calculate_accuracy(self, logits: torch.Tensor, targets: torch.Tensor) -> float:
        mask = targets != self.ignore_index
        predictions = torch.argmax(logits, dim=-1)
        correct = (predictions == targets) & mask
        return correct.sum().item() / mask.sum().item()

    def _train_epoch(self, epoch: int, total_epochs: int) -> Tuple[float, float]:
        self.model.train()
        total_loss = 0.0
        total_accuracy = 0.0
        num_batches = 0

        progress_bar = tqdm(self.train_loader, desc=f"Train Epoch {epoch}/{total_epochs}", leave=False)

        for inputs, targets in progress_bar:
            inputs = inputs.to(self.device)
            targets = targets.to(self.device)

            self.optimizer.zero_grad()
            logits = self.model(inputs)

            loss = self.criterion(logits.view(-1, logits.shape[-1]), targets.view(-1))
            loss.backward()
            self.optimizer.step()

            accuracy = self._calculate_accuracy(logits, targets)

            total_loss += loss.item()
            total_accuracy += accuracy
            num_batches += 1

            progress_bar.set_postfix({"loss": loss.item(), "acc": accuracy})

        return total_loss / num_batches, total_accuracy / num_batches

    def _validate_epoch(self, epoch: int, total_epochs: int) -> Tuple[float, float]:
        self.model.eval()
        total_loss = 0.0
        total_accuracy = 0.0
        num_batches = 0

        progress_bar = tqdm(self.val_loader, desc=f"Val Epoch {epoch}/{total_epochs}", leave=False)

        with torch.no_grad():
            for inputs, targets in progress_bar:
                inputs = inputs.to(self.device)
                targets = targets.to(self.device)

                logits = self.model(inputs)
                loss = self.criterion(logits.view(-1, logits.shape[-1]), targets.view(-1))

                accuracy = self._calculate_accuracy(logits, targets)

                total_loss += loss.item()
                total_accuracy += accuracy
                num_batches += 1

        return total_loss / num_batches, total_accuracy / num_batches

    def fit(self, num_epochs: int) -> List[Dict[str, float]]:
        history = []
        for epoch in range(1, num_epochs + 1):
            train_loss, train_acc = self._train_epoch(epoch, num_epochs)
            val_loss, val_acc = self._validate_epoch(epoch, num_epochs)

            self.writer.add_scalar("Loss/Train", train_loss, epoch)
            self.writer.add_scalar("Accuracy/Train", train_acc, epoch)
            self.writer.add_scalar("Loss/Val", val_loss, epoch)
            self.writer.add_scalar("Accuracy/Val", val_acc, epoch)

            history.append({"train_loss": train_loss, "val_loss": val_loss, "val_acc": val_acc})
        return history

## Experiments

In [11]:
from torch.optim import Adam
import copy

In [12]:
baseline_config = {
    "hidden_size": 128,
    "num_layers": 1,
    "dropout": 0.0,
    "norm_type": "none",
    "tokenizer_type": "word",
    "vocab_size": 10000
}

experiments = {
    "hidden_size": [64, 128, 256],
    "num_layers": [1, 2, 3],
    "dropout": [0.0, 0.05, 0.1],
    "norm_type": ["none", "batch", "layer"],
    "tokenizer_type": ["word", "labse", "char"]
}

def create_dataloaders(config: Dict[str, Any]) -> Tuple[DataLoader, DataLoader, NERUSDataset]:
    dataset = NERUSDataset(
        data=nerus_data,
        tokenizer_type=config["tokenizer_type"],
        vocab_size=config["vocab_size"]
    )

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(
        train_set, batch_size=64, shuffle=True, collate_fn=collate_fn
    )
    val_loader = DataLoader(
        val_set, batch_size=64, shuffle=False, collate_fn=collate_fn
    )

    return train_loader, val_loader, dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss(ignore_index=-100)
num_epochs = 5

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
baseline_writer = SummaryWriter(log_dir="runs/baseline")

train_loader, val_loader, dataset = create_dataloaders(baseline_config)

model = ConfigurableLSTM(
    vocab_size=len(dataset.word2idx),
    num_classes=len(dataset.tag2idx),
    hidden_size=baseline_config["hidden_size"],
    num_layers=baseline_config["num_layers"],
    dropout=baseline_config["dropout"],
    norm_type=baseline_config["norm_type"]
).to(device)

optimizer = Adam(model.parameters(), lr=1e-3)

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    writer=baseline_writer
)

print(f"Training baseline...")
baseline_history = trainer.fit(num_epochs)
baseline_writer.close()

In [ ]:
for param_name, values in experiments.items():
    for value in values:
        if value == baseline_config[param_name]:
            continue

        current_config = copy.deepcopy(baseline_config)
        current_config[param_name] = value

        run_name = f"runs/{param_name}/{value}"
        writer = SummaryWriter(log_dir=run_name)

        train_loader, val_loader, dataset = create_dataloaders(current_config)

        model = ConfigurableLSTM(
            vocab_size=len(dataset.word2idx),
            num_classes=len(dataset.tag2idx),
            hidden_size=current_config["hidden_size"],
            num_layers=current_config["num_layers"],
            dropout=current_config["dropout"],
            norm_type=current_config["norm_type"]
        ).to(device)

        optimizer = Adam(model.parameters(), lr=1e-3)

        trainer = Trainer(
            model=model,
            optimizer=optimizer,
            criterion=criterion,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            writer=writer
        )

        print(f"Training: {param_name} = {value}")
        trainer.fit(num_epochs)
        writer.close()

## DELETE RUNS

In [14]:
!rm -rf runs